# Movie Recommendation System (Content-Based, Genre Similarity)

**What this does:** You type a movie name → it finds movies with similar genres (action, romance, thriller, horror, comedy, etc.) using cosine similarity on genre vectors.

**Approach used:** Content-based filtering only. (Not collaborative filtering / matrix factorization — that needs user rating history, which isn't what this task does. If your project brief asks for "category-based recommendation," this is the correct match. Don't claim collaborative filtering in your report if this is the code you're submitting — a viva examiner will catch that mismatch fast.)

**Dataset:** MovieLens (ml-latest-small) — auto-downloaded below.

## Step 1: Download MovieLens dataset

In [ ]:
import urllib.request
import zipfile
import os

url = "https://files.grouplens.org/datasets/movielens/ml-latest-small.zip"
zip_path = "ml-latest-small.zip"

if not os.path.exists("ml-latest-small"):
    urllib.request.urlretrieve(url, zip_path)
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(".")
    print("Downloaded and extracted.")
else:
    print("Already present.")


## Step 2: Load and inspect data

In [ ]:
import pandas as pd

movies = pd.read_csv("ml-latest-small/movies.csv")
print(movies.shape)
movies.head()


## Step 3: Clean genres

Each movie can have multiple genres (e.g. "Action|Adventure|Sci-Fi"). We keep that — don't force one category per movie, that throws away information and is factually wrong about how the data works.

In [ ]:
# Some movies have genre listed as "(no genres listed)" - drop those, they're useless for this task
movies = movies[movies['genres'] != '(no genres listed)'].reset_index(drop=True)

# Replace | with space so TF-IDF treats each genre as a separate token
movies['genres_clean'] = movies['genres'].str.replace('|', ' ', regex=False)

movies[['title', 'genres', 'genres_clean']].head()


## Step 4: Vectorize genres and compute similarity

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

tfidf = TfidfVectorizer()
genre_matrix = tfidf.fit_transform(movies['genres_clean'])

similarity_matrix = cosine_similarity(genre_matrix)
print(similarity_matrix.shape)


## Step 5: Search + recommend function

User types a movie title (partial match works) → get top N similar movies by genre.

In [ ]:
# map title -> index, lowercase for easy matching
movies['title_lower'] = movies['title'].str.lower()
title_to_index = pd.Series(movies.index, index=movies['title_lower'])

def search_movie(query):
    """Returns list of matching titles for a partial/ fuzzy text search."""
    query = query.lower().strip()
    matches = movies[movies['title_lower'].str.contains(query, na=False)]
    return matches[['title', 'genres']]

def recommend(movie_title, top_n=10):
    movie_title_lower = movie_title.lower().strip()

    if movie_title_lower not in title_to_index:
        # try partial match instead of failing outright
        matches = search_movie(movie_title_lower)
        if matches.empty:
            print(f"No movie found matching '{movie_title}'.")
            return None
        elif len(matches) > 1:
            print(f"Multiple matches found for '{movie_title}'. Be more specific:")
            print(matches['title'].to_string(index=False))
            return None
        else:
            movie_title_lower = matches.iloc[0]['title'].lower()

    idx = title_to_index[movie_title_lower]
    sim_scores = list(enumerate(similarity_matrix[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:top_n+1]  # skip itself (always similarity = 1 with itself)

    rec_indices = [i[0] for i in sim_scores]
    rec_scores = [round(i[1], 3) for i in sim_scores]

    result = movies.iloc[rec_indices][['title', 'genres']].copy()
    result['similarity_score'] = rec_scores
    return result.reset_index(drop=True)


## Step 6: Try it

In [ ]:
recommend("Toy Story (1995)", top_n=10)


In [ ]:
# Try with a partial / unsure title
recommend("dark knight", top_n=5)


In [ ]:
# Try a category-style search instead of an exact movie - search then recommend off the result
search_movie("godfather")


## Notes for your report / viva

- **Why content-based, not collaborative filtering:** Collaborative filtering needs a user-item ratings matrix (who rated what, how). Your task description is "user searches a movie, system suggests similar-genre movies" — that's a property of the movie itself, not user behavior history. Content-based is the structurally correct choice. If a teacher insists you need collaborative filtering too, that's a scope change, not a tweak — say so explicitly rather than bolting on unused code.
- **Limitation to be upfront about:** genre similarity alone is shallow — it ignores plot, actors, director, ratings. Two movies can share "Action|Comedy" and have nothing else in common. If asked "how would you improve this," the honest answer is: add a hybrid layer using collaborative filtering (Surprise library, SVD) once you have a ratings matrix — that's the `ratings.csv` file also included in the MovieLens zip you just downloaded.
- **Don't overclaim "AI"** in your report. This is TF-IDF vectorization + cosine similarity — a statistical method, not a trained AI model. If a viva panel asks "what AI did you use," the honest answer is "this version uses similarity-based filtering, not a trained model; that would be the next step."